# Step 1. Problem statement

[Project setup and results](../README.md) · [Detailed analysis notes](analysis.md)

Download the UCI CSV as described in the project README before running this notebook.

In [ ]:
# Import pandas to load and inspect the dataset.
import pandas as pd

pd.set_option("display.max_columns", None)

In [ ]:
# Load the local UCI CSV from the repository root or notebook directory.
from pathlib import Path

from online_shoppers_intention.data import DEFAULT_DATA

data_path = DEFAULT_DATA if DEFAULT_DATA.is_file() else Path("..") / DEFAULT_DATA
df = pd.read_csv(data_path)

## 1.1. Fixing the holdout before EDA

Reserve a stratified 20% holdout before exploring relationships with purchases. Fix the row indices for reproducible modeling and create `eda_train` from the remaining visits.

In [ ]:
# Fix training and holdout indices before studying relationships with the target.
from online_shoppers_intention.data import TARGET, holdout_indices

train_indices, test_indices = holdout_indices(df[TARGET])
eda_train = df.loc[train_indices].copy()

assert train_indices.intersection(test_indices).empty
assert len(train_indices) + len(test_indices) == len(df)
print(f"Training visits for target-related EDA: {len(eda_train):,}")
print(f"Holdout visits reserved for final evaluation: {len(test_indices):,}")

# Step 2. Exploratory data analysis

In [ ]:
# Import tools for displaying tables and plotting EDA charts.
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from IPython.display import display

## 2.1. Dataset size and structure

In [ ]:
# Show the dataset dimensions and the first five visits.
print(f"Visits: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Features: {df.shape[1] - 1}; target variable: Revenue")
display(df.head())

In [ ]:
# Inspect column types and non-null counts with pandas.
df.info()

## 2.2. Numerical and categorical features

Separate numerical measurements from categorical features according to their meaning. Compare storage types and unique-value counts, including integer codes that represent categories.

In [ ]:
# Group features by meaning and inspect their types and unique counts.
numeric_features = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay",
]
categorical_features = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Weekend",
]

feature_types = pd.DataFrame(
    {"Pandas dtype": df.dtypes.astype(str), "Unique values": df.nunique(dropna=True)}
)
feature_types.index.name = "Feature"

print(f"Numerical features: {len(numeric_features)}")
display(feature_types.loc[numeric_features])
print(f"Categorical features: {len(categorical_features)}")
display(feature_types.loc[categorical_features])

Summarize numerical ranges, quartiles, and zero shares to spot skewed features, unusual values, and meaningful zeros.

In [ ]:
# Inspect ranges, quartiles, and zero shares for numerical features.
numeric_summary = df[numeric_features].describe().T
numeric_summary["Zeros, %"] = df[numeric_features].eq(0).mean().mul(100)
display(numeric_summary.round(3))

Plot numerical distributions on their original scales to reveal long tails, concentrations near zero, and differences in feature ranges.

In [ ]:
# Plot numerical feature histograms on their original scales.
axes = df[numeric_features].hist(
    bins=40,
    figsize=(15, 12),
    layout=(4, 3),
    color="steelblue",
    edgecolor="white",
    grid=False,
)
for ax in axes.flat:
    if ax.get_visible():
        ax.set_xlabel("Value")
        ax.set_ylabel("Number of visits")
        ax.tick_params(axis="x", labelrotation=20)
plt.suptitle("Numerical feature distributions", fontsize=15)
plt.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

Compare category frequencies to identify dominant visitor segments and rare values that may be absent from the training split.

In [ ]:
# Show value frequencies for every categorical feature.
fig, axes = plt.subplots(4, 2, figsize=(15, 14))
for column, ax in zip(categorical_features, axes.flat):
    shares = (
        df[column].value_counts(normalize=True, dropna=False).mul(100).sort_values()
    )
    ax.barh(shares.index.astype(str), shares.to_numpy(), color="steelblue")
    ax.set(title=column, xlabel="Share of visits, %", xlim=(0, 100))
    ax.tick_params(axis="y", labelsize=9)
    ax.grid(axis="x", alpha=0.2)
    ax.set_axisbelow(True)

for ax in axes.flat[len(categorical_features) :]:
    ax.remove()
fig.suptitle("Categorical feature distributions", fontsize=15)
plt.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

## 2.3. Missing values and duplicate rows

Count missing entries in every column and report their percentages to establish whether the current dataset requires imputation.

In [ ]:
# Count missing values and their percentage in each column.
missing_summary = pd.DataFrame(
    {"Missing values": df.isna().sum(), "Missing values, %": df.isna().mean().mul(100)}
)
display(missing_summary.sort_values("Missing values", ascending=False).round(2))

Measure repeated rows and inspect their purchase outcomes to document a potential limitation of the evaluation. Retain the original visits because identical characteristics do not prove that sessions were copied.

In [ ]:
# Check for duplicate rows while preserving the original dataset.
duplicate_mask = df.duplicated()
duplicate_summary = pd.Series(
    {
        "Duplicates after the first occurrence": int(duplicate_mask.sum()),
        "Rows in duplicate groups": int(df.duplicated(keep=False).sum()),
        "Purchases among duplicates after the first occurrence": int(
            df.loc[duplicate_mask, "Revenue"].sum()
        ),
    },
    name="Count",
)
display(duplicate_summary.to_frame())

## 2.4. Target class distribution

Calculate visit counts and percentages for both purchase outcomes to quantify the target imbalance.

In [ ]:
# Count visits and calculate the share of each Revenue class.
class_counts = df["Revenue"].value_counts().reindex([False, True], fill_value=0)
class_distribution = pd.DataFrame(
    {"Number of visits": class_counts, "Share, %": class_counts.div(len(df)).mul(100)}
)
class_distribution.index.name = "Revenue"
display(class_distribution.round(2))

Visualize the class imbalance with annotated purchase and non-purchase counts, making the baseline difficulty clear before comparing models.

In [ ]:
# Compare purchase and non-purchase visit counts in a bar chart.
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(
    ["No purchase (False)", "Purchase (True)"],
    class_counts.to_numpy(),
    color=["steelblue", "seagreen"],
)
ax.bar_label(
    bars,
    labels=[f"{count} ({count / len(df):.2%})" for count in class_counts],
    padding=5,
)
ax.set(
    title="Revenue distribution",
    ylabel="Number of visits",
    ylim=(0, class_counts.max() * 1.17),
)
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## 2.5. Why accuracy alone is insufficient

Evaluate an always-no-purchase rule to demonstrate why high accuracy can be misleading. Compare its accuracy with buyer recall and the number of missed purchases.

In [ ]:
# Calculate accuracy and recall for an always-no-purchase baseline.
no_purchase_predictions = pd.Series(False, index=df.index)
true_positives = int((no_purchase_predictions & df["Revenue"]).sum())
false_negatives = int((~no_purchase_predictions & df["Revenue"]).sum())

accuracy_example = pd.DataFrame(
    {
        "Accuracy": [no_purchase_predictions.eq(df["Revenue"]).mean()],
        "Buyer recall": [true_positives / int(df["Revenue"].sum())],
        "Buyers found (TP)": [true_positives],
        "Buyers missed (FN)": [false_negatives],
    },
    index=["Always predict no purchase"],
)
display(accuracy_example.round(4))

## 2.6. Conversion rate by category

Compare training conversion rates across visitor types, months, traffic sources, and weekdays versus weekends. Show group sizes and 95% Wilson intervals to distinguish well-supported differences from estimates based on few visits.

In [ ]:
# Compare training purchase rates by category with sample sizes and 95% Wilson intervals.
from statistics import NormalDist

from matplotlib.ticker import PercentFormatter


def wilson_interval(successes, totals, confidence=0.95):
    successes = np.asarray(successes, dtype=float)
    totals = np.asarray(totals, dtype=float)
    assert np.all(totals > 0)
    assert np.all((successes >= 0) & (successes <= totals))
    z = NormalDist().inv_cdf(0.5 + confidence / 2)
    proportion = successes / totals
    denominator = 1 + z**2 / totals
    center = (proportion + z**2 / (2 * totals)) / denominator
    half_width = (
        z
        * np.sqrt(proportion * (1 - proportion) / totals + z**2 / (4 * totals**2))
        / denominator
    )
    return np.maximum(0, center - half_width), np.minimum(1, center + half_width)


def summarize_conversion(data, by):
    summary = data.groupby(by, observed=True, sort=False)["Revenue"].agg(
        Visits="size", Purchases="sum"
    )
    summary = summary.loc[summary["Visits"].gt(0)].copy()
    summary["Conversion"] = summary["Purchases"] / summary["Visits"]
    summary["CI lower"], summary["CI upper"] = wilson_interval(
        summary["Purchases"], summary["Visits"]
    )
    assert summary["Visits"].sum() == len(data)
    assert summary["Purchases"].sum() == data["Revenue"].sum()
    assert summary["CI lower"].between(0, 1).all()
    assert summary["CI upper"].between(0, 1).all()
    return summary


train_purchase_rate = eda_train["Revenue"].mean()
month_order = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "June",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]
category_orders = {
    "VisitorType": ["New_Visitor", "Returning_Visitor", "Other"],
    "Month": month_order,
    "TrafficType": sorted(eda_train["TrafficType"].unique()),
    "Weekend": [False, True],
}
category_conversion_tables = {}
fig, axes = plt.subplots(2, 2, figsize=(17, 16), sharex=True)
for column, ax in zip(category_orders, axes.flat):
    summary = summarize_conversion(eda_train, column)
    observed_order = [
        value for value in category_orders[column] if value in summary.index
    ]
    summary = summary.reindex(observed_order)
    category_conversion_tables[column] = summary
    positions = np.arange(len(summary))
    errors = np.vstack(
        [
            summary["Conversion"] - summary["CI lower"],
            summary["CI upper"] - summary["Conversion"],
        ]
    )
    ax.errorbar(
        summary["Conversion"],
        positions,
        xerr=np.maximum(errors, 0),
        fmt="o",
        color="steelblue",
        ecolor="steelblue",
        capsize=3,
        markersize=6,
        linewidth=1.5,
    )
    ax.axvline(train_purchase_rate, color="darkorange", linestyle="--", linewidth=1.5)
    labels = []
    for value, row in summary.iterrows():
        label = (
            ("Weekend" if value else "Weekday") if column == "Weekend" else str(value)
        )
        labels.append(f"{label} (n={int(row['Visits']):,})")
    ax.set_yticks(positions, labels)
    ax.invert_yaxis()
    ax.set(title=column, xlabel="Purchase rate with 95% Wilson interval", xlim=(0, 1))
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.tick_params(axis="x", labelbottom=True)
    ax.grid(axis="x", alpha=0.2)
    ax.set_axisbelow(True)

fig.suptitle(
    f"Conversion by category on train | dashed line: overall {train_purchase_rate:.1%}",
    fontsize=15,
    y=0.995,
)
fig.text(
    0.5,
    0.006,
    "n = number of training visits; wide intervals indicate limited evidence.",
    ha="center",
)
plt.tight_layout(rect=(0, 0.025, 1, 0.975))
plt.show()

## 2.7. Feature distributions by purchase outcome

Compare feature distributions between purchase outcomes using training data and ECDFs normalized within each class. Apply `log1p` to page counts and durations for readability, and report medians on the original scale.

In [ ]:
# Compare class-normalized training ECDFs and medians, using log1p for page counts and durations.
from matplotlib.lines import Line2D

outcome_features = [
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
]
outcome_colors = {False: "steelblue", True: "seagreen"}
outcome_plot_data = eda_train[["Revenue", *outcome_features]].copy()
log_features = ["ProductRelated", "ProductRelated_Duration"]
outcome_plot_data[log_features] = np.log1p(outcome_plot_data[log_features])

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)
for column, ax in zip(outcome_features, axes.flat):
    sns.ecdfplot(
        data=outcome_plot_data,
        x=column,
        hue="Revenue",
        hue_order=[False, True],
        palette=outcome_colors,
        legend=False,
        ax=ax,
    )
    xlabel = f"log1p({column})" if column in log_features else column
    ax.set(
        title=column,
        xlabel=xlabel,
        ylabel="Cumulative share within class",
        ylim=(0, 1.02),
    )
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.grid(alpha=0.2)
    ax.set_axisbelow(True)

outcome_counts = eda_train["Revenue"].value_counts()
handles = [
    Line2D(
        [0],
        [0],
        color=outcome_colors[value],
        linewidth=2,
        label=f"{label} (n={outcome_counts[value]:,})",
    )
    for value, label in [(False, "No purchase"), (True, "Purchase")]
]
fig.suptitle("Training feature distributions by purchase outcome", fontsize=15, y=0.995)
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.965), ncol=2)
plt.tight_layout(rect=(0, 0, 1, 0.925))
plt.show()

outcome_medians = (
    eda_train.groupby("Revenue")[outcome_features]
    .median()
    .rename(index={False: "No purchase", True: "Purchase"})
)
outcome_medians.index.name = "Outcome (train)"
print("Median feature values on the original scale:")
display(outcome_medians.round(4))

## 2.8. Conversion across feature ranges

Explore nonlinear relationships between purchase rates and feature values using training quantile ranges. Separate zero values and show sample sizes with 95% Wilson intervals for each range.

In [ ]:
# Estimate training conversion across ordered quantile ranges, separating zero values.
range_features = ["ProductRelated_Duration", "ExitRates"]
binned_conversion_tables = {}
feature_bin_edges = {}
feature_bin_memberships = {}
for column in range_features:
    values = eda_train[column]
    assert values.notna().all() and values.ge(0).all()
    positive = values.gt(0)
    membership = pd.Series(index=eda_train.index, dtype="object", name="Feature range")
    labels = []
    if values.eq(0).any():
        labels.append("0 (zero)")
        membership.loc[~positive] = labels[0]
    if positive.any():
        if values.loc[positive].nunique() == 1:
            edges = np.array([values.loc[positive].iloc[0]])
            label = f"Value = {edges[0]:.4g}"
            labels.append(label)
            membership.loc[positive] = label
        else:
            codes, edges = pd.qcut(
                values.loc[positive], q=8, labels=False, retbins=True, duplicates="drop"
            )
            range_labels = [
                f"{'[' if index == 0 else '('}{lower:.4g}, {upper:.4g}]"
                for index, (lower, upper) in enumerate(zip(edges[:-1], edges[1:]))
            ]
            labels.extend(range_labels)
            membership.loc[positive] = codes.map(dict(enumerate(range_labels)))
        feature_bin_edges[column] = edges
    else:
        feature_bin_edges[column] = np.array([0.0])
    assert membership.notna().all()
    membership = pd.Series(
        pd.Categorical(membership, categories=labels, ordered=True),
        index=eda_train.index,
        name="Feature range",
    )
    feature_bin_memberships[column] = membership
    summary = summarize_conversion(eda_train, membership).reindex(labels)
    binned_conversion_tables[column] = summary

fig, axes = plt.subplots(1, 2, figsize=(17, 8), sharex=True)
maximum_upper = max(
    table["CI upper"].max() for table in binned_conversion_tables.values()
)
axis_limit = min(1, max(0.35, np.ceil(maximum_upper * 1.1 * 10) / 10))
for column, ax in zip(range_features, axes):
    summary = binned_conversion_tables[column]
    positions = np.arange(len(summary))
    errors = np.vstack(
        [
            summary["Conversion"] - summary["CI lower"],
            summary["CI upper"] - summary["Conversion"],
        ]
    )
    ax.errorbar(
        summary["Conversion"],
        positions,
        xerr=np.maximum(errors, 0),
        fmt="o",
        color="steelblue",
        ecolor="steelblue",
        capsize=4,
        markersize=6,
        linewidth=1.5,
    )
    ax.axvline(train_purchase_rate, color="darkorange", linestyle="--", linewidth=1.5)
    ax.set_yticks(
        positions,
        [f"{label} (n={int(row['Visits']):,})" for label, row in summary.iterrows()],
    )
    ax.invert_yaxis()
    ax.set(
        title=column,
        xlabel="Purchase rate with 95% Wilson interval",
        xlim=(0, axis_limit),
    )
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.grid(axis="x", alpha=0.2)
    ax.set_axisbelow(True)

fig.suptitle(
    "Training conversion across feature ranges | zero group + up to 8 nonzero quantile bins",
    fontsize=15,
    y=0.995,
)
fig.text(
    0.5,
    0.006,
    "Ranges are defined on train only; dashed line shows the overall training purchase rate.",
    ha="center",
)
plt.tight_layout(rect=(0, 0.03, 1, 0.96))
plt.show()

## 2.9. Spearman correlation heatmap

Measure rank correlations among the primary numerical features on train to identify related signals. Exclude category codes, the target, and `PageValues` from the heatmap.

In [ ]:
# Inspect rank correlations on train, excluding the target, category codes, and PageValues.
spearman_features = [column for column in numeric_features if column != "PageValues"]
assert "Revenue" not in spearman_features
assert set(spearman_features).isdisjoint(categorical_features)
spearman_correlation = eda_train[spearman_features].corr(method="spearman")
assert np.allclose(spearman_correlation, spearman_correlation.T)

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(
    spearman_correlation,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Spearman correlation"},
    ax=ax,
)
ax.set_title("Numerical feature correlations on train (without PageValues)", pad=16)
ax.tick_params(axis="y", labelrotation=0)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
plt.tight_layout()
plt.show()

# Step 3. Data splitting and preprocessing

Import the tools needed to combine missing-value handling, categorical encoding, and numerical scaling in reusable preprocessing pipelines.

In [ ]:
# Use the same schema and preprocessing as the training CLI.
from online_shoppers_intention.data import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    PAGE_VALUES,
    split_features_target,
)
from online_shoppers_intention.models import make_preprocessor

## 3.1. Features and target variable

Build the model inputs and binary target while excluding `PageValues` because its availability at prediction time is uncertain. Keep separate numerical and categorical feature lists for preprocessing.

In [ ]:
# Separate the target and exclude PageValues from both models.
X, y = split_features_target(df)
model_numeric_features = list(NUMERIC_FEATURES)
model_categorical_features = list(CATEGORICAL_FEATURES)

print(f"Numerical model features: {len(model_numeric_features)}")
print(model_numeric_features)
print(f"Categorical model features: {len(model_categorical_features)}")
print(model_categorical_features)

## 3.2. Using the fixed stratified split

Construct training and test inputs from the indices fixed before EDA. Verify that modeling uses the same training visits as the target-related exploratory analysis.

In [ ]:
# Apply the fixed train/test indices without creating a new split.
X_train = X.loc[train_indices].copy()
X_test = X.loc[test_indices].copy()
y_train = y.loc[train_indices].copy()
y_test = y.loc[test_indices].copy()

assert X_train.index.equals(eda_train.index)
assert X_test.index.equals(test_indices)

Compare split sizes, purchase counts, and class percentages to confirm that stratification preserved a similar target balance.

In [ ]:
# Check split sizes and purchase shares after stratification.
split_summary = pd.DataFrame(
    {
        "Visits": [len(X_train), len(X_test)],
        "No purchase": [int(y_train.eq(0).sum()), int(y_test.eq(0).sum())],
        "Purchases": [int(y_train.sum()), int(y_test.sum())],
        "Purchase share, %": [100 * y_train.mean(), 100 * y_test.mean()],
    },
    index=["Train", "Test"],
)
display(split_summary.round(3))

Check that train and test share no row indices, then count test visits whose features also occur in train. Distinguish row separation from the presence of identical session characteristics.

In [ ]:
# Check shared indices and matching feature rows between train and test.
shared_indices = X_train.index.intersection(X_test.index)
matching_test_rows = X_test.merge(
    X_train.drop_duplicates(), on=X.columns.tolist(), how="inner"
)
print(f"Shared original indices: {len(shared_indices)}")
print(f"Test visits with identical features in train: {len(matching_test_rows)}")

## 3.3. Preprocessing numerical and categorical features

Create independent preprocessing pipelines for both models: impute numerical values, encode categories, and scale numerical features for logistic regression. Unknown categories are handled when transforming new visits.

In [ ]:
preprocessors = {
    "LogisticRegression": make_preprocessor(scale_numeric=True),
    "DecisionTree": make_preprocessor(scale_numeric=False),
}

## 3.4. Fitting transformations on training data only

Learn preprocessing parameters only from train and apply those fitted transformations to test. Compare the resulting matrix sizes for both model configurations.

In [ ]:
# Fit both preprocessors on train and apply their transformations to test.
train_prepared = {}
test_prepared = {}

for model_name, preprocessor in preprocessors.items():
    train_prepared[model_name] = preprocessor.fit_transform(X_train)
    test_prepared[model_name] = preprocessor.transform(X_test)

prepared_summary = pd.DataFrame(
    [
        {
            "Model": model_name,
            "Train rows": train_prepared[model_name].shape[0],
            "Test rows": test_prepared[model_name].shape[0],
            "Features after preprocessing": train_prepared[model_name].shape[1],
        }
        for model_name in preprocessors
    ]
).set_index("Model")
display(prepared_summary)

Inspect the training-derived imputation medians, means, and scales to make the numerical preprocessing parameters visible.

In [ ]:
# Show imputation and scaling parameters learned only from train.
fitted_numeric_pipeline = preprocessors["LogisticRegression"].named_transformers_[
    "numeric"
]
train_preparation_parameters = pd.DataFrame(
    {
        "Imputation median": fitted_numeric_pipeline.named_steps["imputer"].statistics_,
        "Mean before scaling": fitted_numeric_pipeline.named_steps["scaler"].mean_,
        "Scale (standard deviation)": fitted_numeric_pipeline.named_steps[
            "scaler"
        ].scale_,
    },
    index=model_numeric_features,
)
train_preparation_parameters.index.name = "Numerical feature"
display(train_preparation_parameters.round(3))

Verify split separation, target alignment, excluded features, matching transformed dimensions, and the absence of missing values. These checks confirm that the prepared inputs follow the intended modeling setup.

In [ ]:
# Verify row separation and exclude Revenue and PageValues from model features.
assert X_train.index.intersection(X_test.index).empty
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert set(X.columns) == set(model_numeric_features + model_categorical_features)
assert {TARGET, PAGE_VALUES}.isdisjoint(X.columns)

for model_name, preprocessor in preprocessors.items():
    prepared_feature_names = preprocessor.get_feature_names_out()
    assert not any(
        "PageValues" in name or "Revenue" in name for name in prepared_feature_names
    )
    assert train_prepared[model_name].shape[1] == test_prepared[model_name].shape[1]
    assert pd.DataFrame(train_prepared[model_name]).notna().all().all()
    assert pd.DataFrame(test_prepared[model_name]).notna().all().all()

print(
    "Checks passed: rows are separated, the target and PageValues are excluded, and no missing values remain after preprocessing."
)

# Step 4. Baseline models

Prepare the classifiers, evaluation metrics, and search tools used in the model comparison. The prediction threshold is fixed at 0.5 so both models are evaluated under the same decision rule.

In [ ]:
# Import models, cross-validation tools, metrics, and tuning plot utilities.
from sklearn.base import clone
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    auc,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

from online_shoppers_intention.models import PR_AUC_SCORER, model_searches, training_cv

THRESHOLD = 0.5

Use one evaluation protocol for both classifiers: trapezoidal PR-AUC and five identical stratified folds within train. This makes the hyperparameter searches comparable while keeping test outside the selection process.

In [ ]:
# Use the CLI's PR-AUC scorer, stratified folds and parameter grids.
cv_splits = list(training_cv().split(X_train, y_train))
searches = model_searches(cv_splits, return_train_score=True)
logistic_search = searches["LogisticRegression"]
tree_search = searches["DecisionTree"]

## 4.1. Logistic regression

Establish a logistic regression baseline by combining preprocessing and classification in one pipeline. Fitting the pipeline on train learns the transformations and model parameters together.

In [ ]:
# Fit the logistic regression baseline on train.
logistic_model = clone(logistic_search.estimator).fit(X_train, y_train)
print(f"Logistic regression fitted on {len(X_train)} visits.")

### Tuning `C`

Find a suitable regularization strength using cross-validation within train. Each candidate fits the complete pipeline inside each fold; the best configuration is then refitted on all training rows and its scores are summarized.

In [ ]:
# Tune C by mean CV PR-AUC and refit the best model on all training data.
logistic_search.fit(X_train, y_train)
logistic_model = logistic_search.best_estimator_

lr_results = pd.DataFrame(logistic_search.cv_results_)
lr_tuning_table = pd.DataFrame(
    {
        "C": lr_results["param_classifier__C"].astype(float),
        "Train PR-AUC": lr_results["mean_train_score"],
        "CV PR-AUC": lr_results["mean_test_score"],
        "CV std": lr_results["std_test_score"],
        "CV rank": lr_results["rank_test_score"],
    }
).sort_values("C")

print(f"Best C in the search grid: {logistic_search.best_params_['classifier__C']:g}")
print(f"Mean CV PR-AUC: {logistic_search.best_score_:.4f}")
display(lr_tuning_table.round(4))

Inspect how regularization affects training and validation performance. The plot compares their PR-AUC scores across C values, shows variation between folds, and marks the configuration selected by cross-validation.

In [ ]:
# Plot train and CV PR-AUC against C with variation across folds.
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(lr_tuning_table["C"], lr_tuning_table["Train PR-AUC"], "o-", label="Train")
ax.plot(lr_tuning_table["C"], lr_tuning_table["CV PR-AUC"], "o-", label="CV")
ax.fill_between(
    lr_tuning_table["C"],
    lr_tuning_table["CV PR-AUC"] - lr_tuning_table["CV std"],
    lr_tuning_table["CV PR-AUC"] + lr_tuning_table["CV std"],
    alpha=0.15,
    label="CV ±1 std across folds",
)
ax.axvline(
    logistic_search.best_params_["classifier__C"],
    color="seagreen",
    linestyle="--",
    label=f"Selected C = {logistic_search.best_params_['classifier__C']:g}",
)
ax.set(
    xscale="log",
    xlabel="C: larger → weaker regularization",
    ylabel="PR-AUC",
    title="Tuning logistic regression regularization",
)
ax.grid(alpha=0.2)
ax.legend()
plt.tight_layout()
plt.show()

## 4.2. Decision tree

Establish a decision tree baseline on the same training rows as logistic regression. Its pipeline handles missing values and categories while leaving numerical features unscaled.

In [ ]:
# Fit the decision tree baseline on train without scaling.
tree_model = clone(tree_search.estimator).fit(X_train, y_train)
print(f"Decision tree fitted on {len(X_train)} visits.")

### Tuning tree depth and leaf size

Control tree complexity by searching maximum depth and minimum leaf size on the shared training folds. The best configuration by mean CV PR-AUC is refitted on all train, and the leading candidates are displayed.

In [ ]:
# Tune tree depth and leaf size by CV PR-AUC, then refit the best configuration.
tree_param_grid = tree_search.param_grid
tree_search.fit(X_train, y_train)
tree_model = tree_search.best_estimator_

tree_results = pd.DataFrame(tree_search.cv_results_)
tree_tuning_table = pd.DataFrame(
    {
        "max_depth": tree_results["param_classifier__max_depth"].astype(int),
        "min_samples_leaf": tree_results["param_classifier__min_samples_leaf"].astype(
            int
        ),
        "Train PR-AUC": tree_results["mean_train_score"],
        "CV PR-AUC": tree_results["mean_test_score"],
        "CV std": tree_results["std_test_score"],
        "CV rank": tree_results["rank_test_score"],
    }
)

print(f"Best parameters in the search grid: {tree_search.best_params_}")
print(f"Mean CV PR-AUC: {tree_search.best_score_:.4f}")
display(tree_tuning_table.sort_values("CV rank").head(10).round(4))

Examine the tree search from two perspectives: scores across the full parameter grid and performance by depth at the selected leaf size. The training–validation gap helps reveal overly complex configurations.

In [ ]:
# Show grid CV PR-AUC and train/CV scores by depth at the selected leaf size.
tree_cv_grid = tree_tuning_table.pivot(
    index="max_depth", columns="min_samples_leaf", values="CV PR-AUC"
).reindex(
    index=tree_param_grid["classifier__max_depth"],
    columns=tree_param_grid["classifier__min_samples_leaf"],
)
best_depth = tree_search.best_params_["classifier__max_depth"]
best_leaf = tree_search.best_params_["classifier__min_samples_leaf"]
depth_slice = tree_tuning_table.loc[
    tree_tuning_table["min_samples_leaf"].eq(best_leaf)
].sort_values("max_depth")

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
sns.heatmap(
    tree_cv_grid,
    annot=True,
    fmt=".3f",
    cmap="YlGnBu",
    ax=axes[0],
    cbar_kws={"label": "CV PR-AUC — higher is better"},
)
axes[0].set(
    title="Tuning tree depth and leaf size",
    xlabel="Minimum leaf size",
    ylabel="Maximum depth",
)
axes[1].plot(depth_slice["max_depth"], depth_slice["Train PR-AUC"], "o-", label="Train")
axes[1].plot(depth_slice["max_depth"], depth_slice["CV PR-AUC"], "o-", label="CV")
axes[1].fill_between(
    depth_slice["max_depth"],
    depth_slice["CV PR-AUC"] - depth_slice["CV std"],
    depth_slice["CV PR-AUC"] + depth_slice["CV std"],
    alpha=0.15,
    label="CV ±1 std across folds",
)
axes[1].axvline(best_depth, color="seagreen", linestyle="--", label="Selected depth")
axes[1].set(
    title=f"Depth with min_samples_leaf={best_leaf}",
    xlabel="Maximum depth",
    ylabel="PR-AUC",
)
axes[1].set_xticks(tree_param_grid["classifier__max_depth"])
axes[1].grid(alpha=0.2)
axes[1].legend()
plt.tight_layout()
plt.show()

## 4.3. Test set metrics

Evaluate the tuned models on the reserved test set using both ranking metrics and predictions at the fixed threshold. Probabilities, predictions, and confusion matrices are saved for later comparisons without refitting.

In [ ]:
# Predict test probabilities and labels, then summarize metrics and confusion matrices.
models = {"LogisticRegression": logistic_model, "DecisionTree": tree_model}
test_probas = {}
test_predictions = {}
confusion_matrices = {}
metric_rows = []

for model_name, model in models.items():
    positive_class_index = list(model.classes_).index(1)
    probabilities = model.predict_proba(X_test)[:, positive_class_index]
    predictions = (probabilities >= THRESHOLD).astype(int)
    pr_precision, pr_recall, _ = precision_recall_curve(y_test, probabilities)
    matrix = confusion_matrix(y_test, predictions, labels=[0, 1])

    test_probas[model_name] = probabilities
    test_predictions[model_name] = predictions
    confusion_matrices[model_name] = matrix
    metric_rows.append(
        {
            "Model": model_name,
            "PR-AUC": auc(pr_recall, pr_precision),
            "ROC AUC": roc_auc_score(y_test, probabilities),
            "precision": precision_score(y_test, predictions, zero_division=0),
            "recall": recall_score(y_test, predictions, zero_division=0),
            "F1": f1_score(y_test, predictions, zero_division=0),
            "Confusion matrix [[TN, FP], [FN, TP]]": matrix.tolist(),
        }
    )

test_results = pd.DataFrame(metric_rows).set_index("Model")
print(f"Positive prediction threshold: p >= {THRESHOLD}")
display(test_results.round(4))

## 4.4. Confusion matrix analysis

Make each model's classification errors easy to compare at the fixed threshold. The confusion matrices share a color scale and label actual and predicted purchase outcomes explicitly.

In [ ]:
# Plot both confusion matrices with a shared color scale and explicit class labels.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
max_count = max(matrix.max() for matrix in confusion_matrices.values())

for (model_name, matrix), ax in zip(confusion_matrices.items(), axes):
    ConfusionMatrixDisplay(
        confusion_matrix=matrix, display_labels=["No purchase (0)", "Purchase (1)"]
    ).plot(
        ax=ax,
        cmap="Blues",
        colorbar=False,
        values_format="d",
        im_kw={"vmin": 0, "vmax": max_count},
    )
    ax.set_title(model_name)
    ax.set_xlabel(f"Prediction at p ≥ {THRESHOLD}")
    ax.set_ylabel("Actual class")

plt.tight_layout()
plt.show()

Translate classification errors into quantities relevant to the marketing scenario. The table counts buyers found, buyers missed, unnecessary offers, and total offers implied by each model's test predictions.

In [ ]:
# Translate confusion matrices into buyers found, missed buyers, and unnecessary offers.
action_rows = []
for model_name, matrix in confusion_matrices.items():
    tn, fp, fn, tp = matrix.ravel()
    action_rows.append(
        {
            "Model": model_name,
            "Buyers found (TP)": int(tp),
            "Buyers missed (FN)": int(fn),
            "Unnecessary offers (FP)": int(fp),
            "Total offers (TP + FP)": int(tp + fp),
        }
    )

action_summary = pd.DataFrame(action_rows).set_index("Model")
display(action_summary)

# Step 5. Model selection

## 5.1. Comparing models by the primary metric

Select the final model using the existing mean CV PR-AUC results from train. The code records the winning fitted pipeline and its parameters without using test performance to choose between models.

In [ ]:
# Compare configurations by mean CV PR-AUC and select the final pipeline.
cv_model_rows = []
for model_name, search in searches.items():
    cv_model_rows.append(
        {
            "Model": model_name,
            "Mean CV PR-AUC": search.best_score_,
            "CV std": search.cv_results_["std_test_score"][search.best_index_],
            "Parameters": search.best_params_,
        }
    )

cv_model_comparison = (
    pd.DataFrame(cv_model_rows)
    .set_index("Model")
    .sort_values("Mean CV PR-AUC", ascending=False)
)
selected_model_name = cv_model_comparison["Mean CV PR-AUC"].idxmax()
final_model = models[selected_model_name]

display(cv_model_comparison.round(4))
print(f"Model selected by CV PR-AUC: {selected_model_name}")
print(f"Its parameters: {searches[selected_model_name].best_params_}")
print(
    "final_model is already fitted on all training data; test data was not used for model selection."
)

## 5.2. Out-of-fold predictions for threshold analysis

Generate training probabilities for threshold analysis using the selected model configurations. Each fold refits the complete pipeline on the other folds, so each row's probability comes from a fit that excluded that row.

In [ ]:
# Generate out-of-fold probabilities on train with preprocessing inside each fold.
from sklearn.model_selection import cross_val_predict

oof_probas = {}
for model_name, model in models.items():
    fold_probabilities = cross_val_predict(
        clone(model), X_train, y_train, cv=cv_splits, method="predict_proba"
    )
    oof_probas[model_name] = pd.Series(
        fold_probabilities[:, 1], index=X_train.index, name="Revenue_probability"
    )

print(f"OOF predictions per model: {len(X_train)}")
print(f"Buyers in this dataset: {int(y_train.sum())}")

## 5.3. Buyers and unnecessary offers at different thresholds

Explore predefined thresholds using the saved out-of-fold training probabilities. For each model and threshold, the table reports prediction quality alongside buyers found, missed buyers, and the number of offers.

In [ ]:
# Calculate metrics and offer counts for both OOF predictions at predefined thresholds.
thresholds_to_compare = [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]
threshold_rows = []

for model_name, probabilities in oof_probas.items():
    for threshold in thresholds_to_compare:
        predictions = probabilities.ge(threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_train, predictions, labels=[0, 1]).ravel()
        threshold_rows.append(
            {
                "Model": model_name,
                "Threshold": threshold,
                "precision": precision_score(y_train, predictions, zero_division=0),
                "recall": recall_score(y_train, predictions, zero_division=0),
                "F1": f1_score(y_train, predictions, zero_division=0),
                "Buyers found (TP)": int(tp),
                "Buyers missed (FN)": int(fn),
                "Unnecessary offers (FP)": int(fp),
                "Total offers (TP + FP)": int(tp + fp),
                "Visits receiving an offer, %": 100 * (tp + fp) / len(y_train),
            }
        )

threshold_summary = pd.DataFrame(threshold_rows).set_index(["Model", "Threshold"])
display(threshold_summary.round(3))

Visualize the threshold tradeoff between reaching buyers and sending unnecessary offers. Each model's curve connects the evaluated thresholds, with selected operating points labeled for comparison.

In [ ]:
# Show the tradeoff between buyers found and unnecessary offers as the threshold changes.
fig, ax = plt.subplots(figsize=(9, 5.5))
for model_number, model_name in enumerate(models):
    model_thresholds = threshold_summary.xs(model_name, level="Model")
    (line,) = ax.plot(
        model_thresholds["Unnecessary offers (FP)"],
        model_thresholds["Buyers found (TP)"],
        "o-",
        label=model_name,
    )
    for threshold in [0.1, 0.2, 0.5]:
        row = model_thresholds.loc[threshold]
        ax.annotate(
            f"p ≥ {threshold:g}",
            (row["Unnecessary offers (FP)"], row["Buyers found (TP)"]),
            xytext=(5, 8 if model_number == 0 else -14),
            textcoords="offset points",
            fontsize=8,
            color=line.get_color(),
        )

ax.set(
    title="Threshold tradeoff: OOF predictions on train",
    xlabel="Unnecessary offers (FP)",
    ylabel="Buyers found (TP)",
)
ax.margins(0.12)
ax.grid(alpha=0.2)
ax.legend()
plt.tight_layout()
plt.show()

## 5.4. Interpreting model selection for marketing

Illustrate the marketing tradeoff at threshold 0.2 using concrete out-of-fold counts. The selected model is also compared at 0.2 and 0.5; this example does not establish an optimal business threshold.

In [ ]:
# Explain the tradeoff using OOF buyer counts and unnecessary offers.
ILLUSTRATION_THRESHOLD = 0.2
scenario_summary = threshold_summary.xs(ILLUSTRATION_THRESHOLD, level="Threshold")

for model_name, row in scenario_summary.iterrows():
    print(
        f"{model_name}, threshold {ILLUSTRATION_THRESHOLD}: "
        f"found {int(row['Buyers found (TP)'])} of {int(y_train.sum())} buyers; "
        f"missed {int(row['Buyers missed (FN)'])}; "
        f"unnecessary offers {int(row['Unnecessary offers (FP)'])}; "
        f"total offers {int(row['Total offers (TP + FP)'])}. "
        f"Precision = {row['precision']:.1%}, recall = {row['recall']:.1%}."
    )

selected_scenarios = threshold_summary.xs(selected_model_name, level="Model").loc[
    [ILLUSTRATION_THRESHOLD, THRESHOLD]
]
print(
    f"Selected model: {selected_model_name}; comparison of two thresholds using OOF train predictions:"
)
display(selected_scenarios.round(3))

# Step 6. `PageValues` and data leakage

## 6.1. Why this feature may reveal future outcomes

`PageValues` may use revenue or goal completions that occur after a prediction is made. If so, it reveals information about the future outcome and causes data leakage. Since its availability at scoring time is unverified, it is excluded from the main models and used only in the training diagnostic.

## 6.2. Relationship with the target on training data only

Explore how nonzero PageValues relates to purchases within train. The cross-tabulation reports percentages separately for each outcome, revealing association without establishing whether the feature is available at prediction time.

In [ ]:
# Check the relationship between nonzero PageValues and Revenue using only train rows.
page_values_train = df.loc[X_train.index, "PageValues"].copy()
page_values_target_shares = (
    pd.crosstab(y_train, page_values_train.gt(0), normalize="index")
    .mul(100)
    .rename(columns={False: "PageValues = 0, %", True: "PageValues > 0, %"})
)
page_values_target_shares.columns.name = None
display(page_values_target_shares.round(2))

## 6.3. Diagnostic comparison using cross-validation on training data

Set up an isolated diagnostic experiment for the potentially leaking feature. Independent copies of each model are prepared with and without PageValues, preserving the primary pipelines and final model.

In [ ]:
# Clone the fitted model configurations for a train-only diagnostic.
from sklearn.model_selection import cross_validate

from online_shoppers_intention.models import with_page_values

X_train_with_page_values = X_train.assign(PageValues=page_values_train)
leakage_diagnostic_models = {
    name: {
        "Without PageValues": clone(model),
        "With PageValues (diagnostic)": with_page_values(model),
    }
    for name, model in models.items()
}

Measure the effect of adding PageValues using the same cross-validation folds within train. The code stores fold scores and summarizes their mean and variation without evaluating the diagnostic variants on test.

In [ ]:
# Compare both variants on the same training CV folds without using test.
leakage_cv_rows = []
leakage_cv_fold_scores = {}

for model_name, variants in leakage_diagnostic_models.items():
    for variant_name, diagnostic_pipeline in variants.items():
        diagnostic_X = (
            X_train
            if variant_name == "Without PageValues"
            else X_train_with_page_values
        )
        diagnostic_scores = cross_validate(
            diagnostic_pipeline,
            diagnostic_X,
            y_train,
            cv=cv_splits,
            scoring=PR_AUC_SCORER,
            error_score="raise",
        )
        fold_scores = diagnostic_scores["test_score"]
        leakage_cv_fold_scores[(model_name, variant_name)] = fold_scores.copy()
        leakage_cv_rows.append(
            {
                "Model": model_name,
                "Variant": variant_name,
                "Mean CV PR-AUC": fold_scores.mean(),
                "CV std": fold_scores.std(),
            }
        )

leakage_cv_comparison = pd.DataFrame(leakage_cv_rows).set_index(["Model", "Variant"])
display(leakage_cv_comparison.round(4))

Show how much PageValues changes the diagnostic CV scores for each model. The chart includes variation across folds, and the table reports the gain without treating it as evidence of valid real-time performance.

In [ ]:
# Show the CV PR-AUC change when PageValues is added to diagnostic models only.
variant_order = ["Without PageValues", "With PageValues (diagnostic)"]
leakage_mean_grid = (
    leakage_cv_comparison["Mean CV PR-AUC"]
    .unstack("Variant")
    .reindex(index=list(models), columns=variant_order)
)
leakage_std_grid = (
    leakage_cv_comparison["CV std"]
    .unstack("Variant")
    .reindex(index=list(models), columns=variant_order)
)

fig, ax = plt.subplots(figsize=(10, 5))
leakage_mean_grid.plot.bar(
    ax=ax, yerr=leakage_std_grid, capsize=4, color=["steelblue", "darkorange"], rot=0
)
ax.set(
    title="Diagnostic comparison: CV within train only",
    xlabel="Model",
    ylabel="Mean CV PR-AUC ±1 std",
    ylim=(0, 1),
)
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
ax.legend(title="Feature set", loc="upper left")
plt.tight_layout()
plt.show()

leakage_gain = (
    (
        leakage_mean_grid["With PageValues (diagnostic)"]
        - leakage_mean_grid["Without PageValues"]
    )
    .rename("CV PR-AUC gain with PageValues")
    .to_frame()
)
display(leakage_gain.round(4))

## 6.4. Why a high CV score does not fix data leakage

Check that the leakage experiment has stayed separate from the primary workflow. Assertions confirm that the selected final pipeline and its transformed features still exclude PageValues, while the diagnostic dataset includes it.

In [ ]:
# Verify that the final model and primary feature set still exclude PageValues.
assert final_model is models[selected_model_name]
assert "PageValues" not in X_train.columns
assert "PageValues" not in model_numeric_features
assert "PageValues" not in final_model.feature_names_in_
assert not any(
    "PageValues" in feature_name
    for feature_name in final_model.named_steps["preprocessor"].get_feature_names_out()
)
assert "PageValues" in X_train_with_page_values.columns
print(
    "PageValues is present only in the diagnostic dataset; final_model and its features are unchanged."
)

# Step 7. Conclusions

## 7.1. Precision-recall and ROC curves

Compare both models across prediction thresholds using their saved test probabilities. The paired precision–recall and ROC plots share model colors, include reference baselines, and summarize the corresponding areas without new fitting.

In [ ]:
# Plot PR and ROC curves using saved test probabilities from models without PageValues.
from sklearn.metrics import roc_curve

model_colors = {"LogisticRegression": "steelblue", "DecisionTree": "darkorange"}
test_purchase_share = y_test.mean()
curve_rows = []
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for model_name, probabilities in test_probas.items():
    curve_precision, curve_recall, _ = precision_recall_curve(y_test, probabilities)
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    curve_pr_auc = auc(curve_recall, curve_precision)
    curve_roc_auc = auc(fpr, tpr)
    marker = "o" if model_name == "DecisionTree" else None

    axes[0].plot(
        curve_recall,
        curve_precision,
        color=model_colors[model_name],
        linewidth=2,
        marker=marker,
        markersize=4,
        label=f"{model_name}: PR-AUC={curve_pr_auc:.4f}",
    )
    axes[1].plot(
        fpr,
        tpr,
        color=model_colors[model_name],
        linewidth=2,
        marker=marker,
        markersize=4,
        label=f"{model_name}: ROC AUC={curve_roc_auc:.4f}",
    )
    curve_rows.append(
        {
            "Model": model_name,
            "Test PR-AUC": curve_pr_auc,
            "Test ROC AUC": curve_roc_auc,
        }
    )

axes[0].axhline(
    test_purchase_share,
    color="gray",
    linestyle="--",
    label=f"Test purchase share: {test_purchase_share:.1%}",
)
axes[0].set(title="Test PR curves", xlabel="Recall", ylabel="Precision")
axes[1].plot([0, 1], [0, 1], "--", color="gray", label="Random ranking")
axes[1].set(
    title="Test ROC curves",
    xlabel="False positive rate (FPR)",
    ylabel="True positive rate (recall)",
)

for ax in axes:
    ax.set(xlim=(0, 1), ylim=(0, 1.02))
    ax.grid(alpha=0.2)
axes[0].legend(loc="upper right", fontsize=9)
axes[1].legend(loc="lower right", fontsize=9)
fig.suptitle("Model comparison without PageValues", fontsize=14)
plt.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

curve_scores = pd.DataFrame(curve_rows).set_index("Model")

## 7.2. Model comparison results

Bring the training selection criterion and held-out ranking metrics into one final comparison. The summary identifies the model already selected by CV and reports its fixed parameters without repeating model selection.

In [ ]:
# Combine CV selection scores and test curve metrics without selecting the model again.
conclusion_metrics = (
    cv_model_comparison[["Mean CV PR-AUC"]]
    .rename(columns={"Mean CV PR-AUC": "CV PR-AUC (train)"})
    .join(curve_scores)
)
conclusion_metrics["Selected by CV"] = conclusion_metrics.index == selected_model_name

display(conclusion_metrics.round(4))
print(f"Final model: {selected_model_name}")
print(f"Fixed parameters: {searches[selected_model_name].best_params_}")
print(
    "Primary feature set: 16 features without PageValues; all transformations were fitted on train."
)

## 7.3. Features available at prediction time

Calendar and visit-context features are usually available at scoring time. Page counts and durations must reflect only activity observed so far, while `BounceRates` and `ExitRates` require verified historical statistics available at that moment. `PageValues` and `Revenue` are excluded from model inputs. Completed-session data supports offline evaluation but does not establish real-time performance.

## 7.4. Project summary

A depth-2 decision tree without `PageValues` is selected by training CV PR-AUC. At threshold 0.5 it misses all test buyers, so choosing an offer threshold still requires business costs and analysis on train. Real-time performance needs validation with feature snapshots, and the incremental benefit of discounts requires a controlled experiment.